# Idea 7: Patient Rescue Radar — EDA & Feature Engineering
> **เป้าหมาย:** ทำความเข้าใจ pattern การขาดนัดใน EMR และสร้าง LTFU prediction prototype  
> **แนวคิดหลัก:** ข้อมูลที่ *หายไป* (missingness) คือ signal สำคัญที่สุด ไม่ใช่ noise ที่ต้องแก้ไข

---
**สมาชิกทีม:** กอ, แบงค์, จีน, ข้าวฟาง, เฟิม — KMUTT  
**วันที่สร้าง:** 31 พ.ค. 2569

## Section 0: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    import missingno as msno
    HAS_MISSINGNO = True
except ImportError:
    HAS_MISSINGNO = False
    print("missingno ไม่ได้ติดตั้ง — รัน: pip install missingno")

try:
    import xgboost as xgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, roc_auc_score
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost/sklearn ไม่ได้ติดตั้ง — รัน: pip install xgboost scikit-learn")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap ไม่ได้ติดตั้ง — รัน: pip install shap")

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')
print("✓ Import สำเร็จ")

In [ ]:
import os

# ---- Config: Path ----
def _find_data_root():
    """ค้นหา Sampled Dataset จาก candidate anchors — ทำงานได้ทั้งใน VS Code และ command line"""
    candidates = []
    try:
        candidates.append(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)))
    except NameError:
        pass
    candidates.append(os.getcwd())
    candidates.append(os.path.dirname(os.getcwd()))
    target = os.path.join("Sampled Dataset", "bdi-hackathon-2026-sampled-dataset")
    tried = []
    for anchor in candidates:
        candidate_path = os.path.join(anchor, target)
        tried.append(candidate_path)
        if os.path.isdir(candidate_path):
            return candidate_path
    raise FileNotFoundError(
        "ไม่พบ Sampled Dataset ที่ path ใดเลย\n" +
        "\n".join(f"  - {p}" for p in tried)
    )

_DATA_ROOT = _find_data_root()
print(f"✓ DATA_ROOT: {_DATA_ROOT}")

DIABETES_XLSX     = os.path.join(_DATA_ROOT, "diabetes",
                                  "data_dictionary_diabetes_example.xlsx")
HYPERTENSION_XLSX = os.path.join(_DATA_ROOT, "hypertension",
                                  "data_dictionary_hypertension_example.xlsx")

# Clinical threshold สำหรับ full dataset (70K คน): 3 periods = ~6 เดือน
# Section 3 จะ auto-adjust ให้สูงขึ้นอัตโนมัติถ้า sample data ทำให้ทุกคนเป็น LTFU
LTFU_GAP_THRESHOLD = 3
RANDOM_SEED = 42

# ---- Helper: โหลด xlsx พร้อม auto-detect sheet ----
def load_patient_data(xlsx_path, label):
    try:
        xl = pd.ExcelFile(xlsx_path)
        print(f"[{label}] Sheets ที่พบ: {xl.sheet_names}")

        best_sheet, best_df, best_rows = None, None, -1
        for sheet in xl.sheet_names:
            try:
                df_try = pd.read_excel(xlsx_path, sheet_name=sheet, header=0)
                if len(df_try) > best_rows:
                    best_rows, best_df, best_sheet = len(df_try), df_try, sheet
            except Exception:
                pass

        if best_df is not None and best_rows > 0:
            print(f"[{label}] ✓ ใช้ sheet: '{best_sheet}' — {best_df.shape[0]} rows × {best_df.shape[1]} columns")
            return best_df
        else:
            print(f"[{label}] ✗ ไม่พบ sheet ที่มีข้อมูล")
            return None
    except FileNotFoundError:
        print(f"[{label}] ✗ ไม่พบไฟล์: {xlsx_path}")
        print(f"         ตรวจสอบ path: {os.path.abspath(xlsx_path)}")
        return None
    except Exception as e:
        print(f"[{label}] ✗ Error: {e}")
        return None

# ---- โหลดข้อมูล ----
df_dm = load_patient_data(DIABETES_XLSX, "DM")
df_ht = load_patient_data(HYPERTENSION_XLSX, "HT")

## Section 1: Data Overview
ทำความเข้าใจโครงสร้างข้อมูล — column groups และ period range

In [ ]:
def summarize_column_groups(df, label):
    """แยก columns ตาม group และแสดงสรุป"""
    groups = {
        'demographics': [c for c in df.columns if c in ['age','sex','identify_by','dm_onset','ht_onset','type1','type2','gdm']],
        'vitalsign':    [c for c in df.columns if c.startswith('vitalsign_')],
        'lab':          [c for c in df.columns if c.startswith('lab_')],
        'comorbidity':  [c for c in df.columns if c.startswith('co_')],
        'medication':   [c for c in df.columns if c.startswith('med_')],
    }
    print(f"\n{'='*50}")
    print(f" Dataset: {label}  ({df.shape[0]} patients × {df.shape[1]} columns)")
    print(f"{'='*50}")
    for g, cols in groups.items():
        print(f"  {g:15s}: {len(cols):4d} columns")
    
    # ดึง period numbers ที่มีจาก vitalsign_sbp_P
    sbp_cols = [c for c in df.columns if 'vitalsign_sbp_' in c]
    if sbp_cols:
        periods = sorted([int(c.split('_')[-1]) for c in sbp_cols])
        print(f"\n  Period range (vitalsign_sbp): P={periods[0]} ถึง P={periods[-1]}  ({len(periods)} periods)")
    return groups

if df_dm is not None:
    groups_dm = summarize_column_groups(df_dm, 'Diabetes (DM)')
if df_ht is not None:
    groups_ht = summarize_column_groups(df_ht, 'Hypertension (HT)')

In [ ]:
# แสดง sample ผู้ป่วย 3 คน (demographics + SBP ช่วงต้น)
if df_dm is not None:
    demo_cols = [c for c in ['age','sex','identify_by','dm_onset','type1','type2'] if c in df_dm.columns]
    sbp_sample = [c for c in df_dm.columns if 'vitalsign_sbp_' in c][:6]
    display_cols = demo_cols + sbp_sample
    print("ตัวอย่าง 5 rows (DM dataset):")
    print(df_dm[display_cols].head())

## Section 2: Missingness Analysis

> **แนวคิดหลักของ Idea 7:** ความแหว่ง (missingness) ใน EMR ไม่ใช่ข้อผิดพลาดของข้อมูล  
> แต่คือ **fingerprint ของพฤติกรรมผู้ป่วย** — คนที่หยุดมาโรงพยาบาลจะไม่มีข้อมูล vitalsign บันทึก  
> เราต้อง *วัด* และ *เข้าใจ* pattern นี้ก่อนสร้าง model

In [ ]:
def compute_group_missingness(df, groups):
    """คำนวณ missing rate ต่อ column group"""
    results = {}
    for g, cols in groups.items():
        if cols:
            missing_rate = df[cols].isnull().mean().mean() * 100
            results[g] = round(missing_rate, 1)
    return results

if df_dm is not None:
    miss_dm = compute_group_missingness(df_dm, groups_dm)
    miss_ht = compute_group_missingness(df_ht, groups_ht) if df_ht is not None else {}

    fig, ax = plt.subplots(figsize=(9, 4))
    labels = list(miss_dm.keys())
    x = np.arange(len(labels))
    width = 0.35
    bars1 = ax.bar(x - width/2, [miss_dm.get(l, 0) for l in labels], width, label='DM', color='#3498db', alpha=0.85)
    if miss_ht:
        bars2 = ax.bar(x + width/2, [miss_ht.get(l, 0) for l in labels], width, label='HT', color='#e74c3c', alpha=0.85)
    ax.set_ylabel('Missing Rate (%)')
    ax.set_title('Missing Rate ต่อ Column Group (DM vs HT)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 105)
    ax.legend()
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{bar.get_height():.0f}%', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    print("\nDM missing rates:", miss_dm)
    if miss_ht:
        print("HT missing rates:", miss_ht)

In [ ]:
# Heatmap: Patient × Period — มี vitalsign_sbp หรือเปล่า
if df_dm is not None:
    sbp_cols = sorted([c for c in df_dm.columns if 'vitalsign_sbp_' in c],
                      key=lambda x: int(x.split('_')[-1]))
    if sbp_cols:
        presence = df_dm[sbp_cols].notna().astype(int)
        presence.columns = [c.replace('vitalsign_sbp_', 'P') for c in sbp_cols]

        fig, ax = plt.subplots(figsize=(min(len(sbp_cols)*0.6 + 2, 18), 6))
        sns.heatmap(presence, cmap=['#f8d7da','#c3e6cb'], linewidths=0.3,
                    linecolor='#dee2e6', cbar=False, ax=ax,
                    xticklabels=True, yticklabels=False)
        ax.set_title('Patient × Period — SBP มีข้อมูล (เขียว) / ไม่มี (แดง)', fontsize=12)
        ax.set_xlabel('Period (P)')
        ax.set_ylabel('ผู้ป่วย (แต่ละแถว = 1 คน)')
        plt.tight_layout()
        plt.show()
        print(f"แผนภาพแสดง {len(df_dm)} ผู้ป่วย × {len(sbp_cols)} periods")
        print("สังเกต: ผู้ป่วยบางคนมีช่องว่างยาวต่อเนื่อง = สัญญาณ LTFU ที่ model จะ learn")

In [ ]:
# missingno matrix (ถ้ามี library)
if HAS_MISSINGNO and df_dm is not None:
    sbp_cols = [c for c in df_dm.columns if 'vitalsign_sbp_' in c]
    hba1c_cols = [c for c in df_dm.columns if 'lab_hba1c_' in c]
    med_cols = [c for c in df_dm.columns if c.startswith('med_')][:5]
    sample_cols = sbp_cols[:10] + hba1c_cols[:5] + med_cols
    
    fig, ax = plt.subplots(figsize=(14, 5))
    msno.matrix(df_dm[sample_cols], ax=ax, sparkline=False, color=(0.2, 0.5, 0.8))
    ax.set_title('Missingno Matrix — SBP (ซ้าย), HbA1c (กลาง), Medication (ขวา)', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# จำนวน periods ที่ผู้ป่วยแต่ละคนมี SBP recorded
if df_dm is not None:
    sbp_cols = [c for c in df_dm.columns if 'vitalsign_sbp_' in c]
    if sbp_cols:
        visit_counts = df_dm[sbp_cols].notna().sum(axis=1)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].hist(visit_counts, bins=15, color='#2ecc71', edgecolor='white', linewidth=0.8)
        axes[0].set_xlabel('จำนวน periods ที่มี SBP recorded')
        axes[0].set_ylabel('จำนวนผู้ป่วย')
        axes[0].set_title('Distribution ของ Visit Count per Patient (DM)')
        axes[0].axvline(visit_counts.median(), color='#e74c3c', linestyle='--', label=f'Median = {visit_counts.median():.0f}')
        axes[0].legend()

        # Scatter: visit_count vs age
        if 'age' in df_dm.columns:
            axes[1].scatter(df_dm['age'], visit_counts, alpha=0.6, color='#3498db', edgecolors='none')
            axes[1].set_xlabel('อายุ (ปี)')
            axes[1].set_ylabel('จำนวน periods ที่มี SBP')
            axes[1].set_title('อายุ vs จำนวน Visit (DM)')
        
        plt.tight_layout()
        plt.show()
        print(f"\nMedian visit periods: {visit_counts.median():.0f} | Min: {visit_counts.min()} | Max: {visit_counts.max()}")
        print(f"ผู้ป่วยที่มี < 3 periods: {(visit_counts < 3).sum()} คน ({(visit_counts < 3).mean()*100:.0f}%)")

## Section 3: LTFU Label Definition

เราต้องนิยาม "Lost-to-Follow-Up" จากข้อมูลที่มี  
**นิยามที่เลือก:** ผู้ป่วยที่มี **≥ 3 consecutive periods** ที่ไม่มีข้อมูล `vitalsign_sbp` บันทึก

> ทำไม 3 periods? 1 period ≈ 60 วัน → 3 periods = **~6 เดือน** ไม่มาโรงพยาบาลเลย  
> ในทางคลินิก: ผู้ป่วย NCD ที่หายไป 6 เดือน = ความเสี่ยงสูงมากที่ BP/glucose จะ rebound

In [ ]:
def compute_max_consecutive_gap(row, sbp_cols_sorted):
    """นับ consecutive missing periods ยาวสุดของผู้ป่วย 1 คน"""
    presence = row[sbp_cols_sorted].notna().values  # True = มีข้อมูล
    max_gap = 0
    current_gap = 0
    for v in presence:
        if not v:
            current_gap += 1
            max_gap = max(max_gap, current_gap)
        else:
            current_gap = 0
    return max_gap

def get_last_observed_period(row, sbp_cols_sorted, periods_sorted):
    """หา period สุดท้ายที่มีข้อมูล vitalsign"""
    vals = row[sbp_cols_sorted]
    non_null_periods = [p for p, v in zip(periods_sorted, vals) if pd.notna(v)]
    return max(non_null_periods) if non_null_periods else -99

if df_dm is not None:
    sbp_cols_dm = sorted(
        [c for c in df_dm.columns if 'vitalsign_sbp_' in c],
        key=lambda x: int(x.split('_')[-1])
    )
    periods_dm = [int(c.split('_')[-1]) for c in sbp_cols_dm]

    df_dm['max_consecutive_gap'] = df_dm.apply(
        lambda row: compute_max_consecutive_gap(row, sbp_cols_dm), axis=1
    )
    df_dm['last_observed_period'] = df_dm.apply(
        lambda row: get_last_observed_period(row, sbp_cols_dm, periods_dm), axis=1
    )
    df_dm['visit_frequency'] = df_dm[sbp_cols_dm].notna().sum(axis=1) / len(sbp_cols_dm)

    print(f"คำนวณ max_consecutive_gap สำเร็จ")
    print(df_dm[['max_consecutive_gap','last_observed_period','visit_frequency']].describe().round(2))

In [ ]:
# กำหนด LTFU label พร้อม Auto-detect threshold สำหรับ sample data
if df_dm is not None and 'max_consecutive_gap' in df_dm.columns:

    # --- Extended Sensitivity Analysis ---
    # รวม threshold สูงขึ้นเพื่อดูการกระจายใน sample data ที่มี gap สูง
    gap_p25 = int(np.percentile(df_dm['max_consecutive_gap'], 25))
    gap_p50 = int(np.percentile(df_dm['max_consecutive_gap'], 50))
    gap_p75 = int(np.percentile(df_dm['max_consecutive_gap'], 75))
    gap_p90 = int(np.percentile(df_dm['max_consecutive_gap'], 90))
    extended_thresholds = sorted(set([3, 10, gap_p25, gap_p50, gap_p75, gap_p75 + 1, gap_p90]))

    print("Sensitivity Analysis — LTFU threshold:\n")
    print(f"  {'Threshold':>14} | {'(~เดือน)':>10} | {'LTFU Count':>10} | {'LTFU %':>8}")
    print("  " + "-"*52)
    for t in extended_thresholds:
        n_ltfu = (df_dm['max_consecutive_gap'] >= t).sum()
        pct = n_ltfu / len(df_dm) * 100
        months = round(t * 60 / 30)
        marker = " ← clinical default (full data)" if t == 3 else ""
        print(f"  >= {t:3d} periods  (~{months:4d} เดือน)  | {n_ltfu:>10} | {pct:>7.0f}%{marker}")

    # --- Auto-detect threshold ---
    # ถ้า clinical threshold (3) ทำให้ทุกคนเป็น LTFU → ปรับ threshold อัตโนมัติ
    # โดยใช้ 75th percentile + 1 → ได้ class balance ~75% Active / 25% LTFU
    effective_threshold = LTFU_GAP_THRESHOLD
    if (df_dm['max_consecutive_gap'] >= LTFU_GAP_THRESHOLD).all():
        effective_threshold = gap_p75 + 1
        print(f"\nℹ️  Sample data auto-adjust:")
        print(f"   - min gap = {df_dm['max_consecutive_gap'].min():.0f} periods → threshold {LTFU_GAP_THRESHOLD} ทำให้ LTFU = 100%")
        print(f"   - ปรับเป็น threshold = {effective_threshold} (75th percentile + 1)")
        print(f"   - ผล: Active ~75% / LTFU ~25% เหมาะสำหรับทดสอบ model")
        print(f"   - หมายเหตุ: threshold จริงสำหรับ full dataset = 3 periods (~6 เดือน)")
    else:
        print(f"\nใช้ clinical threshold = {effective_threshold} periods (~{effective_threshold*2} เดือน)")

    # กำหนด label จริง
    df_dm['ltfu_label'] = (df_dm['max_consecutive_gap'] >= effective_threshold).astype(int)
    n_ltfu = df_dm['ltfu_label'].sum()
    n_active = len(df_dm) - n_ltfu

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Active (ติดตามได้)', 'LTFU (ขาดนัด)'], [n_active, n_ltfu],
           color=['#2ecc71', '#e74c3c'], edgecolor='white')
    ax.set_ylabel('จำนวนผู้ป่วย')
    ax.set_title(f'LTFU Label Distribution (threshold = {effective_threshold} periods)')
    for i, v in enumerate([n_active, n_ltfu]):
        ax.text(i, v + 0.3, str(v), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"\nจำนวน LTFU: {n_ltfu} ({n_ltfu/len(df_dm)*100:.0f}%) | Active: {n_active} ({n_active/len(df_dm)*100:.0f}%)")

## Section 4: Feature Engineering

สร้าง features ทั้งหมดที่ใช้ predict LTFU  
**ทุก feature derive ได้จาก EMR โดยตรง** — ไม่ต้องใช้ข้อมูลภายนอก

In [ ]:
def compute_slope(series_values, period_indices):
    """คำนวณ slope ของ y ตาม x โดย skip NaN  — คืน NaN ถ้าข้อมูลน้อยเกินไป"""
    pairs = [(p, v) for p, v in zip(period_indices, series_values) if pd.notna(v)]
    if len(pairs) < 2:
        return np.nan
    xs = np.array([p[0] for p in pairs], dtype=float)
    ys = np.array([p[1] for p in pairs], dtype=float)
    if xs.std() == 0:
        return 0.0
    return np.polyfit(xs, ys, 1)[0]  # slope

if df_dm is not None:
    # --- SBP features ---
    sbp_cols_dm_sorted = sorted(
        [c for c in df_dm.columns if 'vitalsign_sbp_' in c],
        key=lambda x: int(x.split('_')[-1])
    )
    periods_dm_sorted = [int(c.split('_')[-1]) for c in sbp_cols_dm_sorted]

    # SBP ที่ last visit
    def get_last_sbp(row):
        vals = [(p, row[c]) for p, c in zip(periods_dm_sorted, sbp_cols_dm_sorted) if pd.notna(row[c])]
        return vals[-1][1] if vals else np.nan
    df_dm['sbp_at_last_visit'] = df_dm.apply(get_last_sbp, axis=1)

    # SBP slope (ใช้ทุก period ที่มีข้อมูล)
    df_dm['sbp_slope'] = df_dm[sbp_cols_dm_sorted].apply(
        lambda row: compute_slope(row.values, periods_dm_sorted), axis=1
    )

    # --- HbA1c features ---
    hba1c_cols_dm = sorted(
        [c for c in df_dm.columns if 'lab_hba1c_' in c],
        key=lambda x: int(x.split('_')[-1])
    )
    hba1c_periods = [int(c.split('_')[-1]) for c in hba1c_cols_dm]

    if hba1c_cols_dm:
        df_dm['lab_sparsity_ratio'] = df_dm[hba1c_cols_dm].isnull().mean(axis=1)
        df_dm['hba1c_slope'] = df_dm[hba1c_cols_dm].apply(
            lambda row: compute_slope(row.values, hba1c_periods), axis=1
        )
        df_dm['hba1c_at_baseline'] = df_dm.get('lab_hba1c_0', np.nan)
    else:
        df_dm['lab_sparsity_ratio'] = np.nan
        df_dm['hba1c_slope'] = np.nan
        df_dm['hba1c_at_baseline'] = np.nan

    # --- Medication features ---
    med_cols_dm = [c for c in df_dm.columns if c.startswith('med_')]
    if med_cols_dm:
        def med_in_last_periods(row, n=3):
            last_p = row.get('last_observed_period', -99)
            relevant = [c for c in med_cols_dm if any(
                c.endswith(f'_{p}') for p in range(max(-1, last_p - n*2), last_p + 1)
            )]
            if not relevant:
                return 0
            return int(row[relevant].notna().any())
        df_dm['med_in_last_period'] = df_dm.apply(med_in_last_periods, axis=1)
    else:
        df_dm['med_in_last_period'] = np.nan

    # --- Comorbidity count ---
    co_cols_dm = [c for c in df_dm.columns if c.startswith('co_')]
    if co_cols_dm:
        df_dm['comorbidity_count'] = df_dm[co_cols_dm].notna().sum(axis=1)
    else:
        df_dm['comorbidity_count'] = 0

    # --- Encode identify_by ---
    if 'identify_by' in df_dm.columns:
        df_dm['identify_by_enc'] = df_dm['identify_by'].astype('category').cat.codes
    else:
        df_dm['identify_by_enc'] = 0

    print("✓ Feature engineering เสร็จสมบูรณ์")
    feature_cols = [
        'visit_frequency', 'max_consecutive_gap', 'last_observed_period',
        'sbp_at_last_visit', 'sbp_slope',
        'lab_sparsity_ratio', 'hba1c_slope', 'hba1c_at_baseline',
        'med_in_last_period', 'comorbidity_count',
        'age', 'identify_by_enc'
    ]
    existing_feats = [f for f in feature_cols if f in df_dm.columns]
    print(f"Features สร้างได้: {len(existing_feats)} / {len(feature_cols)}")
    print(df_dm[existing_feats].describe().round(3))

## Section 5: Exploratory Analysis — Features vs LTFU

ดูว่า features ที่สร้างขึ้นมามี pattern ต่างกันระหว่างกลุ่ม LTFU และ Active จริงหรือไม่

In [ ]:
if df_dm is not None and 'ltfu_label' in df_dm.columns:
    ltfu_group = df_dm[df_dm['ltfu_label'] == 1]
    active_group = df_dm[df_dm['ltfu_label'] == 0]

    compare_feats = ['visit_frequency', 'max_consecutive_gap', 'sbp_at_last_visit',
                     'lab_sparsity_ratio', 'comorbidity_count', 'age']
    compare_feats = [f for f in compare_feats if f in df_dm.columns]

    n_feats = len(compare_feats)
    ncols = 3
    nrows = (n_feats + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
    axes = axes.flatten()

    for i, feat in enumerate(compare_feats):
        ax = axes[i]
        ltfu_vals = ltfu_group[feat].dropna()
        active_vals = active_group[feat].dropna()

        ax.boxplot([active_vals, ltfu_vals],
                   labels=['Active', 'LTFU'],
                   patch_artist=True,
                   boxprops=dict(facecolor='#d4edda' if feat != 'max_consecutive_gap' else '#f8d7da'),
                   medianprops=dict(color='black', linewidth=2))
        ax.set_title(feat, fontsize=10)
        ax.set_ylabel('ค่า')

        if len(active_vals) > 0 and len(ltfu_vals) > 0:
            ax.text(1, active_vals.median(), f' Med={active_vals.median():.2f}', va='center', fontsize=8, color='#27ae60')
            ax.text(2, ltfu_vals.median(), f' Med={ltfu_vals.median():.2f}', va='center', fontsize=8, color='#e74c3c')

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Feature Distribution: Active vs LTFU (DM dataset)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap ระหว่าง features
if df_dm is not None:
    feat_for_corr = [f for f in existing_feats if f in df_dm.columns and df_dm[f].notna().sum() > 5]
    corr_matrix = df_dm[feat_for_corr + ['ltfu_label']].corr()

    fig, ax = plt.subplots(figsize=(11, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
                annot_kws={'size': 8})
    ax.set_title('Correlation Matrix — Features + LTFU Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    ltfu_corr = corr_matrix['ltfu_label'].drop('ltfu_label').sort_values(key=abs, ascending=False)
    print("\nCorrelation กับ ltfu_label (เรียงตาม |r|):")
    print(ltfu_corr.round(3).to_string())

In [ ]:
# Patient Profile Summary
if df_dm is not None and 'ltfu_label' in df_dm.columns:
    profile_feats = [f for f in existing_feats if f in df_dm.columns]
    profile = df_dm.groupby('ltfu_label')[profile_feats].median().round(2).T

    col_map = {0: 'Active (0)', 1: 'LTFU (1)'}
    profile.columns = [col_map.get(c, str(c)) for c in profile.columns]

    if 'Active (0)' in profile.columns and 'LTFU (1)' in profile.columns:
        profile['ทิศทาง'] = profile.apply(
            lambda row: '↑ LTFU สูงกว่า' if row['LTFU (1)'] > row['Active (0)'] else '↓ LTFU ต่ำกว่า', axis=1
        )
    else:
        present = list(profile.columns)
        print(f"หมายเหตุ: พบเพียง class {present} ในข้อมูลนี้ — ปรับ LTFU_GAP_THRESHOLD ถ้าต้องการเห็นทั้งสอง class")

    print("Median Feature Profile: Active vs LTFU")
    print(profile.to_string())

## Section 6: Quick XGBoost Prototype (Proof of Concept)

> **ข้อควรระวัง:** n = 100 คน เท่านั้น — ตัวเลข performance ที่ได้ยังไม่ reflect ความเป็นจริง  
> จุดประสงค์ของ section นี้คือ **validate ทิศทางของ features** และแสดง SHAP importance  
> ใช้กับ dataset เต็ม 70K/150K คน onsite จึงจะได้ตัวเลขจริง

In [ ]:
if HAS_XGB and df_dm is not None and 'ltfu_label' in df_dm.columns:
    feat_cols_xgb = [f for f in existing_feats if f in df_dm.columns and f != 'ltfu_label']
    X = df_dm[feat_cols_xgb].copy()
    y = df_dm['ltfu_label'].copy()

    mask = y.notna()
    X, y = X[mask], y[mask]

    if len(y) < 10:
        print("Sample น้อยเกินไปสำหรับ train — ข้าม Section 6")
    elif y.nunique() < 2:
        print("ไม่มี class หนึ่งในข้อมูล (LTFU = 0 หมด หรือ 1 หมด) — ปรับ threshold ใน Section 3")
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y if y.value_counts().min() >= 2 else None
        )

        scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_SEED,
            eval_metric='logloss',
            verbosity=0
        )
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        print("=== Classification Report ===")
        print(classification_report(y_test, y_pred, target_names=['Active', 'LTFU']))
        if len(np.unique(y_test)) > 1:
            auc = roc_auc_score(y_test, y_proba)
            print(f"AUC-ROC: {auc:.3f}")
        else:
            print("AUC: ไม่สามารถคำนวณได้ (test set มี class เดียว — เพิ่ม data หรือปรับ threshold)")

        importances = pd.Series(model.feature_importances_, index=feat_cols_xgb).sort_values(ascending=True)
        fig, ax = plt.subplots(figsize=(8, max(4, len(feat_cols_xgb)*0.4)))
        importances.plot.barh(ax=ax, color='#3498db', edgecolor='white')
        ax.set_title('XGBoost Feature Importance (built-in gain)')
        ax.set_xlabel('Importance Score')
        plt.tight_layout()
        plt.show()

else:
    print("XGBoost ไม่พร้อม หรือข้อมูลไม่เพียงพอ — ติดตั้ง xgboost ก่อน")

In [ ]:
# SHAP Analysis
if HAS_SHAP and HAS_XGB and 'model' in dir() and 'X_train' in dir():
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train)

    plt.figure(figsize=(10, 5))
    shap.summary_plot(shap_values, X_train, plot_type='bar', show=False, max_display=12)
    plt.title('SHAP Feature Importance — Top Features ที่ Drive LTFU Prediction')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    shap.summary_plot(shap_values, X_train, show=False, max_display=12)
    plt.title('SHAP Beeswarm — ทิศทาง feature ต่อ LTFU (แดง = ค่าสูง, น้ำเงิน = ค่าต่ำ)')
    plt.tight_layout()
    plt.show()
    print("\nตีความ SHAP:")
    print("- Features ที่อยู่บนสุด = สำคัญที่สุดในการทำนาย LTFU")
    print("- แดง + บวก = ค่าสูง → เพิ่มความเสี่ยง LTFU")
    print("- น้ำเงิน + ลบ = ค่าต่ำ → ลดความเสี่ยง LTFU")
else:
    print("SHAP ไม่พร้อม — ติดตั้ง shap หรือรัน Section 6 ก่อน")

## Section 7: Summary & Next Steps

### สิ่งที่ค้นพบจาก EDA นี้

**Missingness Pattern:**
- Vitalsign (SBP/DBP) มี missingness สูง — แต่ distribution ไม่สม่ำเสมอ → บางคนมาสม่ำเสมอ บางคนหายไปยาว
- Lab (HbA1c, FPG) มี missingness สูงกว่า vitalsign เพราะ ordered เฉพาะกรณี → sparsity ของ lab เป็น signal ของ visit behavior
- Medication ถ้าบันทึกไม่ครบ = สัญญาณว่าผู้ป่วยอาจไม่ได้รับยา

**LTFU Definition:**
- ใช้ ≥ 3 consecutive missing vitalsign periods (≈ 6 เดือนไม่มาโรงพยาบาล)
- Threshold นี้สอดคล้องกับคำนิยาม clinical LTFU ในงานวิจัยสุขภาพ

**Feature Signal:**
- `visit_frequency` และ `max_consecutive_gap` คาดว่ามี discriminative power สูงสุด
- `sbp_slope` และ `hba1c_slope` อาจ capture deterioration ก่อน dropout
- `comorbidity_count` อาจสัมพันธ์แบบ U-shape (คนหนักมากไม่มา, คนเบาก็ไม่รู้สึกจำเป็นต้องมา)

### ข้อควรระวัง (Bias & Limitations)

| ข้อควรระวัง | คำอธิบาย | Mitigation |
|---|---|---|
| Missing ≠ LTFU จริง | อาจแค่ย้ายสิทธิ/ย้ายโรงพยาบาล | บอก case manager ให้ verify ก่อนโทรหา |
| Sample size n=100 | AUC ที่ได้ไม่ reliable | ใช้ dataset เต็ม onsite |
| Label imbalance | LTFU อาจมีน้อย → bias | ใช้ scale_pos_weight ใน XGBoost (ทำแล้ว) |
| PDPA | ข้อมูลผู้ป่วยเป็น sensitive data | output เป็น risk score เท่านั้น ไม่ expose raw data |

### Next Steps สำหรับ Onsite Hackathon

1. **โหลด dataset เต็ม** 70K (DM) + 150K (HT) — รัน pipeline เดิมนี้ได้เลย
2. **Tune LTFU threshold** ร่วมกับ clinical team (อาจปรับเป็น 2 หรือ 4 periods ขึ้นกับ clinical judgment)
3. **เพิ่ม feature:** DBP slope, medication class change (จาก monotherapy → polypharmacy = อาการหนักขึ้น)
4. **Cross-validate:** Stratified K-Fold เพื่อ estimate AUC ที่ reliable กว่า
5. **สร้าง output:** ranked list ของผู้ป่วยเสี่ยง LTFU พร้อม risk score → ส่งให้ Case Manager / อสม.
6. **เตรียมพิตช์:** แสดง SHAP plot เป็น "3 เหตุผลหลักที่ผู้ป่วยคนนี้เสี่ยง" ภาษาคลินิก